In [ ]:
# Install the required libraries
!pip install -qU pydantic langchain langchain-openai faiss-cpu langchain_community

**Importing Packages**

In [ ]:
import pandas as pd
import numpy as np
import json
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from pathlib import Path
import os
from getpass import getpass

**Managing Directories Paths**

In [ ]:
# Set all path variables
PROJECT_PATH = Path("/content/capstone_project/")
POLICIES_DIR = PROJECT_PATH / "data" / "policies"
POLICY_FAISS_INDEX_PATH = PROJECT_PATH / "outputs" / "vector_stores" / "policy_faiss_index"

In [ ]:
# Prompt for the OpenAI API Key securely
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

# Optional: required only for institutional/proxy endpoints
base_url = getpass(
    "Enter OPENAI_BASE_URL (press Enter if not required): "
).strip()

if base_url:
    os.environ["OPENAI_BASE_URL"] = base_url
else:
    os.environ.pop("OPENAI_BASE_URL", None)

**Loading of the Retail Policies JSON**

In [ ]:
policies_json_path = POLICIES_DIR / "retail_policies.json"
with open(policies_json_path, "r", encoding="utf-8") as f:
    policies_json_data = json.load(f)

**Creation of LangChain Policies Document Objects**

In [ ]:
documents = []
for doc_id, text in policies_json_data.items():
    doc = Document(
        page_content=text,
        metadata={"doc_id": doc_id}  # Stores the key as metadata for tracking
    )
    documents.append(doc)

**Initializing Embedding model**

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

**Initializing Vector store**

In [ ]:
# 4. Initialize and populate the vector store
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)
print(f"Successfully stored {len(documents)} documents.")

**Initializing the Retriever**

In [ ]:
def retrieve_policy(query: str):
  # Converting vector store into the Retriever
  retriever = vector_store.as_retriever()

  # Generate the retreiver context
  retrieved_context = retriever.invoke(query)

  return retrieved_context

In [ ]:
# Time to verify if policy retriever return the expected response
retriver = retrieve_policy("Can I return shoes after 30 days?")
print(retriver[0].page_content)

**Saving the Policy FAISS Index**

In [ ]:
vector_store.save_local(POLICY_FAISS_INDEX_PATH)
print(f"Files saved directly inside: {POLICY_FAISS_INDEX_PATH}")

# **Conclusion And Outcome**

In this notebook, I have implemented the Policy RAG component of the Retail Assistant for answering policy-related customer questions.

**Activities Performed:**

* Prepared the retail policy documents for retrieval.
* Created embeddings for policy content.
* Built the policy FAISS vector index.
* Implemented semantic retrieval of relevant policy information.
* Tested the component using representative policy-related queries.

**Output :**

Policy FAISS index: outputs/vector_stores/policy_faiss_index/



